# Cross Validation und Feature selection with CatBoost

In [ ]:
from sklearn.model_selection import train_test_split, TimeSeriesSplit
import sklearn.metrics as metrics
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
#from sklearn.metrics import mean_absolute_percentage_error
from scipy.stats import randint, uniform, loguniform
from scipy.stats import uniform as sp_randFloat
from scipy.stats import randint as sp_randInt  
import catboost
from catboost import *
import shap
import numpy as np
import pandas as pd
import matplotlib.pylab as pl
import matplotlib.pyplot as plt
from pandas import read_csv
import plotly.graph_objects as go
import time
import seaborn as sns
import re

In [ ]:
def regression_results(y_true, y_pred):
    # Regression metrics
    explained_variance=metrics.explained_variance_score(y_true, y_pred)
    mean_absolute_error=metrics.mean_absolute_error(y_true, y_pred) 
    mse=metrics.mean_squared_error(y_true, y_pred) 
    #mean_squared_log_error=metrics.mean_squared_log_error(y_true, y_pred)
    median_absolute_error=metrics.median_absolute_error(y_true, y_pred)
    r2=metrics.r2_score(y_true, y_pred)
    # mape = mean_absolute_percentage_error(y_true, y_pred)

    print('explained_variance: ', round(explained_variance,4))
    # print('mean_squared_log_error: ', round(mean_squared_log_error,4))
    print('r2: ', round(r2,4))
    print('MAE: ', round(mean_absolute_error,4))
    print('MSE: ', round(mse,4))
    print('RMSE: ', round(np.sqrt(mse),4))
    print("Dataframe SHAPE:",y_true.shape, y_pred.shape)
    # print('MAPE: ',round(mape,4))
    # print(y_true.head(5), y_pred.head(5))
    # print(y_true.describe())
    # print(y_pred.describe())

    # Mean Arctangent Absolute Percentage Error (MAAPE)
    # Source: https://gist.github.com/bshishov/5dc237f59f019b26145648e2124ca1c9
    EPSILON = 1e-10
    maape = np.mean(np.arctan(np.abs((y_true - y_pred) / (y_pred + EPSILON))))
    print('MAAPE: ',round(maape,4))

# print the JS visualization code to the notebook
shap.initjs()

## Load dataset

In [ ]:
%%time
csv_dataset="Radius_200m"
dataframe = read_csv('../../Daten/Solar_Load_Profile/2018_Features_Solar_Load_Profile_' + csv_dataset + '_all_SEVIRI.csv')
print('Describe Dataframe:')
print(dataframe.describe())
print(dataframe.shape)
counts = dataframe['TIMESTAMP'].value_counts(dropna=False)
min_count = counts.min()
max_count = counts.max()
min_list = counts[counts == min_count].index.tolist()
max_list = counts[counts == max_count].index.tolist()
print(f"Datei: {csv_dataset}")
print(f"\nMaximale Anzahl je TIMESTAMP: {max_count}")
print("TIMESTAMP(s) mit maximaler Anzahl:")
for ts in max_list:
    print("  -", ts)
duplikate_ohne_id = dataframe[dataframe.iloc[:, 1:].duplicated(keep=False)]
print(duplikate_ohne_id)
dataframe.rename(columns={'GLOBAL_RADIATION_SENSOR_VALUE': 'SENSOR_VALUE',
                          'GLOBAL_RADIATION_SENSOR_TEMPERATURE': 'SENSOR_TEMPERATURE',
                          'MAX_EINSPEISELEISTG': 'PV_INSTALL',
                          'SOLAR_RADIATION_GLOBALRAD': 'GLOBALRAD',
                          'SOLAR_RADIATION_DIRECTRAD':'DIRECTRAD',
                          'SOLAR_RADIATION_DIFFUSERAD':'DIFFUSERAD'}, inplace=True)


## Calculate additional Features

In [ ]:
dataframe['HOUR_DEZ'] = pd.to_datetime(dataframe['TIMESTAMP']).dt.hour + (pd.to_datetime(dataframe['TIMESTAMP']).dt.minute/60)
dataframe['DAY_YEAR'] = pd.to_datetime(dataframe['TIMESTAMP']).dt.dayofyear
dataframe['BTD'] = dataframe['IR39'] - dataframe['IR108']
dataframe['LOAD'] = (dataframe['SOLAR_LOAD_VALUE'] / dataframe['PV_INSTALL']) * 1000
dataframe['DISTANCE'] = (dataframe['GLOBAL_RADIATION_SENSOR_DISTANCE'] / 1000)
print('Describe Dataframe:')
print(dataframe.head(5))
print(dataframe.describe())
print(dataframe.shape)
features = dataframe[["HEIGHT","PV_INSTALL","GLOBALRAD","VIS06","IR39","WV62","IR108","HOUR_DEZ","DAY_YEAR"]]
cbm_filename ="Model_Solar_Load_Profile_" + csv_dataset + "_WMS_SEVIRI_Bands_WGS84_N200.cbm"
target = dataframe['LOAD']
print('Describe features:')
print(features.describe())
print(features.shape)
print('Describe target:')
print(target.describe())
print(target.shape)
X_display,y_display = features,target
X,y = features,target
for col in features.columns:
    print(col)
print(X.shape)
print(y.shape)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

### Intial Hyperparameter Optimizing with Rolling-Window Cross-Validation

In [ ]:
%%time
start_time = time.time()
tscv = TimeSeriesSplit(n_splits=12)
n_iter = 2
best_score = np.inf
best_params = {}
best_iter = 0
print("--- Starting manual Random Search with time-series cross-validation ---")
print(f"Number of iterations: {n_iter}")
print(f"Number of CV splits per iteration: {tscv.get_n_splits()}\n")
for i in range(n_iter):
    params = {
        'depth': randint(5, 9).rvs(),
        'l2_leaf_reg': loguniform(1e-1, 10).rvs(),
        'iterations': randint(500, 1500).rvs(),
        'learning_rate': loguniform(0.02, 0.3).rvs(),
        'bagging_temperature': uniform(0.0, 3.0).rvs(),
        'bootstrap_type': 'Bayesian'
    }
    fold_scores = []
    start_iter_time = time.time()
    for train_idx, val_idx in tscv.split(X_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        model = CatBoostRegressor(
            loss_function='RMSE', eval_metric='RMSE', od_type='Iter', od_wait=20,
            random_seed=55, task_type='GPU', has_time=True, verbose=0,
            allow_writing_files=False, border_count=128, **params
        )
        model.fit(X_tr, y_tr, eval_set=(X_val, y_val), use_best_model=True, verbose=False)
        preds = model.predict(X_val)
        rmse = np.sqrt(np.mean((y_val - preds)**2))
        fold_scores.append(rmse)
    end_iter_time = time.time()
    avg_rmse = np.mean(fold_scores)
    iter_duration = end_iter_time - start_iter_time
    if avg_rmse < best_score:
        best_score = avg_rmse
        best_params = params.copy()
        best_iter = i + 1
    formatted_params = ', '.join(f'{k}={v:.4f}' if isinstance(v, float) else f'{k}={v}' for k, v in params.items())
    timestamp = time.strftime('%Y-%m-%d %H:%M:%S', time.localtime())
    print(f"Time: {timestamp} | Iter{i+1:3d}/{n_iter} | Duration:{iter_duration:6.2f}s | CV RMSE: {avg_rmse:.4f} | Best RMSE: {best_score:.4f} ({best_iter}) | Params: {formatted_params}")
end_time = time.time()
total_duration_minutes = (end_time - start_time) / 60
print("\n--- Search completed ---")
print(f"Total duration: {total_duration_minutes:.2f} minutes")
print("\nBest found parameters:")
for key, value in best_params.items():
    if isinstance(value, float):
        print(f"  '{key}': {value:.6f}")
    else:
        print(f"  '{key}': {value}")
print(f"\nBest RMSE score: {best_score:.4f} (achieved at iteration {best_iter})")

# Classic feature attributions

In [ ]:
%%time
print("Parameters Passed When Creating Model : ",model.get_params())
print("\nAll Model Parameters                : ",model.get_all_params())
print("\nBest Score                  : ",model.get_best_score())
print("\nCategorical Feature Indices : ",model.get_cat_feature_indices())
print("\nFeature Importances        : ",model.get_feature_importance())
print("\nCount of trees in model = {}".format(model.tree_count_))

### Save Model as CBM-File

In [ ]:
model.save_model(cbm_filename, format="cbm", export_parameters=None, pool=None)

# Make Predictions

In [ ]:
%%time
y_pred = model.predict(X)
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)
print('Describe Prediction:')
print(y_pred.shape)
print(pd.DataFrame(y_pred).describe())
print(pd.DataFrame(y_pred).head(5))
print(y_train_pred.shape)
print(y_test_pred.shape)

In [ ]:
print("Results Training Data")
regression_results(y_train, y_train_pred)
print('')
print("Results Validation Data")
regression_results(y_test, y_test_pred)

In [ ]:
%%time
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X)

In [ ]:
shap.summary_plot(shap_values, X_display, plot_type = "bar" )

In [ ]:
%%time
shap.summary_plot(shap_values, X_display)